### Phase 4: Importing and pulling weather report details for analysis
Input: Precipitation and Temperature readings from NOAA of 2025

In [1]:
!pip install noaa-sdk pandas

In [2]:
from google.colab import auth
auth.authenticate_user()

In [5]:
# Import the data tables
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tmaxst-v1.0.0-20260406
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tminst-v1.0.0-20260406
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-pcpnst-v1.0.0-20260406

--2026-04-20 15:24:13--  https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tmaxst-v1.0.0-20260406
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.172, 205.167.25.168, 205.167.25.177, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.172|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1255380 (1.2M)
Saving to: ‘climdiv-tmaxst-v1.0.0-20260406.1’

climdiv-tmaxst-v1.0 100%[===================>]   1.20M  2.74MB/s    in 0.4s    

2026-04-20 15:24:14 (2.74 MB/s) - ‘climdiv-tmaxst-v1.0.0-20260406.1’ saved [1255380/1255380]

--2026-04-20 15:24:14--  https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tminst-v1.0.0-20260406
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.172, 205.167.25.168, 205.167.25.177, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.172|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1255380 (1.2M)
Saving to: ‘climdiv-tminst-v1.0.0-20260406’

climdiv

In [6]:
import pandas as pd

state_code_map = {
    "001": "AL", "002": "AZ", "003": "AR", "004": "CA", "005": "CO",
    "006": "CT", "007": "DE", "008": "FL", "009": "GA", "010": "ID",
    "011": "IL", "012": "IN", "013": "IA", "014": "KS", "015": "KY",
    "016": "LA", "017": "ME", "018": "MD", "019": "MA", "020": "MI",
    "021": "MN", "022": "MS", "023": "MO", "024": "MT", "025": "NE",
    "026": "NV", "027": "NH", "028": "NJ", "029": "NM", "030": "NY",
    "031": "NC", "032": "ND", "033": "OH", "034": "OK", "035": "OR",
    "036": "PA", "037": "RI", "038": "SC", "039": "SD", "040": "TN",
    "041": "TX", "042": "UT", "043": "VT", "044": "VA", "045": "WA",
    "046": "WV", "047": "WI", "048": "WY", "050": "AK"
}

def parse_statewide_file(filename, value_name):
    rows = []

    with open(filename, "r") as f:
        for line in f:
            state_code = line[0:3]
            division_flag = line[3]  # should be '0' for statewide
            year = int(line[6:10])

            if division_flag != "0":
                continue

            if state_code not in state_code_map:
                continue

            if year != 2025:
                continue

            state = state_code_map[state_code]

            for i in range(12):
                start = 10 + i * 7
                raw = line[start:start+7].strip()

                if raw in ["-99.90", "-9.99", "-9999."]:
                    value = None
                else:
                    value = float(raw)

                rows.append({
                    "state": state,
                    "year_month": int(f"{year}{i+1:02d}"),
                    value_name: value
                })

    return pd.DataFrame(rows)

The above step imports pandas, but also converts the weather data formatting into something that is easier to understand. The NOAA files identify states using numeric area codes rather than two-letter abbreviations. By changing it, we can specifically join the weather data to the spend data later.

Since the NOAA files are not CSVs, we need to convert them into a more reasonable format for our research purposes. Each line currently. The function reads each line, keeps only the statewide rows for 2025 (since the data from Aramark is only from 2025), extracts the 12 monthly values, and turns them into a tidy dataframe stores metadata plus 12 monthly values in a fixed-width format.

In [7]:
tmax = parse_statewide_file("climdiv-tmaxst-v1.0.0-20260406", "tmax_f")
tmin = parse_statewide_file("climdiv-tminst-v1.0.0-20260406", "tmin_f")
precip = parse_statewide_file("climdiv-pcpnst-v1.0.0-20260406", "total_precip_inches")

#Computer average temperature, rather than having max and min.
temp = tmax.merge(tmin, on=["state", "year_month"], how="inner")
temp["avg_temp_f"] = (temp["tmax_f"] + temp["tmin_f"]) / 2
temp = temp[["state", "year_month", "avg_temp_f"]]

# Merge into one Weather table, for easy access
weather = temp.merge(precip, on=["state", "year_month"], how="inner")
weather = weather[["state", "year_month", "avg_temp_f", "total_precip_inches"]]
weather.head()

,state,year_month,avg_temp_f,total_precip_inches
0,AL,202501,40.10,3.24
1,AL,202502,52.70,4.45
2,AL,202503,58.25,5.20
3,AL,202504,66.80,5.65
4,AL,202505,72.00,9.83


Because we downloaded maximum and minimum temperature rather than direct average temperature, this step estimates average temperature. On top of that, we merge the precipitation values into one table. This makes it more useful and reasonable to work with in future phases.

In [22]:
warm_months = weather[weather["avg_temp_f"] > 50]

rain_threshold = (
    warm_months
    .groupby("state")["total_precip_inches"]
    .quantile(0.25)
)

rain_threshold = rain_threshold.clip(lower=0.2)

weather["rain_threshold"] = weather["state"].map(rain_threshold)
weather["snow_proxy"] = (
    (weather["avg_temp_f"] <= 32) &
    (
        (weather["total_precip_inches"] >= weather["rain_threshold"]) |
        (weather["total_precip_inches"] > 0.1)
    )
).astype(int)

print(weather.groupby("snow_proxy")["avg_temp_f"].mean())
print(weather["snow_proxy"].value_counts())


snow_proxy
0    58.423360
1    22.613529
Name: avg_temp_f, dtype: float64
snow_proxy
0    503
1     85
Name: count, dtype: int64


Now it is not possible to get a specific measure of snowfall from the states, as most data summaries merge snowfall with precipitation. Yet snow storms, are a big reason why distribution can be impacted, which can impact the sales that occur. To account for this, a temporary snow estimation setup has been created.

Assuming the conditions for snowfall are temperature below or at freezing, we can parse through the average precipitation during the warm months. This sets a threshhold which can predict additional/replacement snowfall on top of this. This threshold also serves to point out abnormal periods of rain/storms that may occur over time.

The computed rain thresholds are mapped back to the full dataset so that each row is evaluated relative to its own state’s climate conditions rather than using a single global threshold. Eventually a prediction of snow can be accumulated, which is then merged with the weather table for further analysis of the situation.

In [26]:
final_weather = weather[
    ["state", "year_month", "avg_temp_f", "total_precip_inches", "rain_threshold", "snow_proxy"]
].copy()

print(final_weather.head())
print(final_weather.dtypes)
print("Rows:", len(final_weather))
print("States:", final_weather["state"].nunique())
print("Months:", final_weather["year_month"].nunique())

  state  year_month  avg_temp_f  total_precip_inches  rain_threshold  \
0    AL      202501       40.10                 3.24            3.01   
1    AL      202502       52.70                 4.45            3.01   
2    AL      202503       58.25                 5.20            3.01   
3    AL      202504       66.80                 5.65            3.01   
4    AL      202505       72.00                 9.83            3.01   

   snow_proxy  
0           0  
1           0  
2           0  
3           0  
4           0  
state                   object
year_month               int64
avg_temp_f             float64
total_precip_inches    float64
rain_threshold         float64
snow_proxy               int64
dtype: object
Rows: 588
States: 49
Months: 12


In [25]:
from google.cloud import bigquery

table_id = "big-data-algorithms-493312.aramark_spend.weather_state_monthly"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

load_job = client.load_table_from_dataframe(
    final_weather,
    table_id,
    job_config=job_config
)

load_job.result()

print(f"Upload complete: {table_id}")

Upload complete: big-data-algorithms-493312.aramark_spend.weather_state_monthly
